In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.ticker import MultipleLocator
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
from astropy import units as u
from reproject import reproject_from_healpix, reproject_to_healpix
from astropy_healpix import HEALPix
from astropy.coordinates import SkyCoord
from astropy.coordinates import Galactic, ICRS
import healpy as hp
import gc
from scipy import interpolate

In [ ]:
hduQ_dragons = fits.open('/srv/data/dva/dragons_hoard/Data_cubes_Jun_2025_BB/convolved_masked_cubes/350to1000/conv_hpx/q_350to1000MHz_convolved_3.6deg_masked_gal_K.hpx.fits')
hduU_dragons = fits.open('/srv/data/dva/dragons_hoard/Data_cubes_Jun_2025_BB/convolved_masked_cubes/350to1000/conv_hpx/u_350to1000MHz_convolved_3.6deg_masked_gal_K.hpx.fits')
hduQ_lbs = fits.open('/srv/data/gmims/gmims-lbs/2025/reprocess-2025-06-26/AO_gal_test/q_lbs_gal.hpx.fits')
hduU_lbs = fits.open('/srv/data/gmims/gmims-lbs/2025/reprocess-2025-06-26/AO_gal_test/u_lbs_gal.hpx.fits')


In [ ]:
Q_dragons = hduQ_dragons[0].data
U_dragons = hduU_dragons[0].data
Q_lbs = hduQ_lbs[0].data
U_lbs = hduU_lbs[0].data


In [ ]:
print(Q_dragons.shape)
print(U_dragons.shape)
print(Q_lbs.shape)
print(U_lbs.shape)


In [ ]:
freqs_dragons = WCS(hduQ_dragons[0].header).all_pix2world(0,range(Q_dragons.shape[0]),0)[1]
freqs_lbs = WCS(hduQ_lbs[0].header).all_pix2world(0,range(Q_lbs.shape[0]),0)[1]


In [ ]:
fpick = 408.

idx_dragons = abs(fpick - freqs_dragons/1e6).argmin()
idx_lbs     = abs(fpick - freqs_lbs/1e6).argmin()

print(freqs_dragons[idx_dragons])
print(freqs_lbs[idx_lbs])

print(idx_dragons)
print(idx_lbs)

In [ ]:
hp.cartview(Q_dragons[idx_dragons],min=-1.5,max=1.5,cmap='RdBu_r')
hp.graticule()
hp.cartview(Q_lbs[idx_lbs],min=-1.5,max=1.5,cmap='RdBu_r')
hp.graticule()

In [ ]:
Q_lbs_408 = hp.pixelfunc.ud_grade(Q_lbs[idx_lbs], 128)
U_lbs_408 = hp.pixelfunc.ud_grade(U_lbs[idx_lbs], 128)

In [ ]:
fig,ax = plt.subplots(1,2,figsize=(12,4))

ax[0].scatter(Q_dragons[idx_dragons], Q_lbs_408,s=0.05)
ax[1].scatter(U_dragons[idx_dragons], U_lbs_408,s=0.05)

ax[0].set_xlabel('DRAGONS Q')
ax[0].set_ylabel('GMIMS-LBS Q')

ax[1].set_xlabel('DRAGONS U')
ax[1].set_ylabel('GMIMS-LBS U')

for i in range(0,2):
    ax[i].set_xlim(-3,3)
    ax[i].set_ylim(-3,3)

plt.savefig('/home/aordog/Dropbox/plots/gmims/lbs_dragons_June2025/dragons_lbs_408.png')

In [ ]:
freqs_dragons = WCS(hduQ_dragons[0].header).all_pix2world(0,0,range(Q_dragons.shape[0]),0)[2]
dec_dragons   = WCS(hduQ_dragons[0].header).all_pix2world(0,range(Q_dragons.shape[1]),0,0)[1]
ra_dragons    = WCS(hduQ_dragons[0].header).all_pix2world(range(Q_dragons.shape[2]),0,0,0)[0]

freqs_lbs = WCS(hduQ_lbs[0].header).all_pix2world(0,0,range(Q_lbs.shape[0]),0)[2]
dec_lbs   = WCS(hduQ_lbs[0].header).all_pix2world(0,range(Q_lbs.shape[1]),0,0)[1]
ra_lbs    = WCS(hduQ_lbs[0].header).all_pix2world(range(Q_lbs.shape[2]),0,0,0)[0]


In [ ]:
idx_bad = np.where(dec_lbs > 20)

print(idx_bad)

In [ ]:
fpick = 408.

idx_dragons = abs(fpick - freqs_dragons/1e6).argmin()
idx_lbs     = abs(fpick - freqs_lbs/1e6).argmin()

print(freqs_dragons[idx_dragons])
print(freqs_lbs[idx_lbs])


In [ ]:
JytoK_all = np.load('/srv/data/dva/conversion_factor_JytoK_July2024_BB.npy')
print(JytoK_all[0].shape)

f = interpolate.interp1d(JytoK_all[0], JytoK_all[1])
JytoK = f(freqs_dragons/1e6)

fig, ax = plt.subplots(1,1,figsize=(10,2))
ax.scatter(JytoK_all[0],JytoK_all[1], s=1)
ax.scatter(freqs_dragons/1e6, JytoK, s=1)

In [ ]:
beam_coefs = [-1.00373215e-08 , 2.66732709e-05 ,-2.47155372e-02 , 9.39383209e+00]
dragons_beam = np.polyval(beam_coefs, freqs_dragons[idx_dragons]/1e6)
print(dragons_beam)

In [ ]:
lbs_beam = 80./60.
fwhm = np.sqrt(dragons_beam**2 - lbs_beam**2)
print(fwhm)

wcs = WCS(hduQ_lbs[0].header).dropaxis(2)

Q_lbs_hpx, footprint = reproject_to_healpix((Q_lbs[idx_lbs],wcs),coord_system_out='C',nside=256)
Q_lbs_hpx[np.isnan(Q_lbs_hpx)] = hp.UNSEEN
Q_lbs_hpx[~np.isfinite(Q_lbs_hpx)] = hp.UNSEEN
Q_lbs_hpx_conv = hp.sphtfunc.smoothing(Q_lbs_hpx, fwhm=np.radians(fwhm))
Q_lbs_conv, footprint = reproject_from_healpix((Q_lbs_hpx_conv,'icrs'),wcs, nested=False)

U_lbs_hpx, footprint = reproject_to_healpix((U_lbs[idx_lbs],wcs),coord_system_out='C',nside=256)
U_lbs_hpx[np.isnan(U_lbs_hpx)] = hp.UNSEEN
U_lbs_hpx[~np.isfinite(U_lbs_hpx)] = hp.UNSEEN
U_lbs_hpx_conv = hp.sphtfunc.smoothing(U_lbs_hpx, fwhm=np.radians(fwhm))
U_lbs_conv, footprint = reproject_from_healpix((U_lbs_hpx_conv,'icrs'),wcs, nested=False)

In [ ]:
def make_the_plot(map_plot, wcs, cbar_label, vmin=-2, vmax=2):

    fs = 30

    sb_cmap = sns.color_palette('RdBu_r', as_cmap=True)
    crange1 = SkyCoord([24,0],  [0,0], frame=ICRS, unit=(u.hourangle, u.deg))
    crange2 = SkyCoord([12,12], [-90,90], frame=ICRS, unit=(u.hourangle, u.deg))
    sb_cmap.set_bad(color='gray')
    
    fig = plt.figure(figsize=(30,13))
    plt.subplots_adjust(left=0.02, bottom=0.1, right=0.99, top=0.94)
    #plt.rcParams['mathtext.fontset'] = 'stix'
        
    gs = gridspec.GridSpec(ncols=1, nrows=1, figure=fig)
        
    ax = fig.add_subplot(gs[0], projection=wcs.celestial)

    im = ax.imshow(map_plot,origin='lower',cmap=sb_cmap,vmin=vmin,vmax=vmax)

    ax.set_ylim(wcs.world_to_pixel(crange2)[1])
    ax.set_xlim(wcs.world_to_pixel(crange1)[0])
    ax.set_xlabel(' ')
    ax.set_ylabel('Declination',fontsize=fs)
    ax.set_xlabel('Right Ascension',fontsize=fs)
    ax.tick_params(axis='both', labelsize=fs, which='both', width=2, length=6)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)

    cbar = fig.colorbar(im, ax=ax, shrink=1, pad=0.01, aspect=15)
    cbar.set_label(cbar_label, fontsize=fs, labelpad=10)
    cbar.ax.tick_params(labelsize=fs, length=6, width=2)
    for spine in cbar.ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)

    return

In [ ]:
wcs = WCS(hduQ_dragons[0].header).dropaxis(2)

make_the_plot(U_dragons[idx_dragons]*JytoK_all[1][idx_dragons], wcs, r'$U$ (K)', vmin=-1, vmax=1)

plt.savefig('/home/aordog/Dropbox/plots/gmims/lbs_dragons_June2025/dragons_U.png')

In [ ]:
wcs = WCS(hduQ_lbs[0].header).dropaxis(2)

make_the_plot(U_lbs[idx_lbs], wcs, r'$U$ (K)', vmin=-1, vmax=1)

plt.savefig('/home/aordog/Dropbox/plots/gmims/lbs_dragons_June2025/lbs_U.png')

In [ ]:
wcs = WCS(hduQ_lbs[0].header).dropaxis(2)

make_the_plot(U_lbs_conv, wcs, r'$U$ (K)', vmin=-1, vmax=1)

plt.savefig('/home/aordog/Dropbox/plots/gmims/lbs_dragons_June2025/lbs_conv_U.png')